<a href="https://colab.research.google.com/github/NiloyKumarKundu/Graph-Neural-Network/blob/main/4_Recurrent_Neural_Network_(RNN).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

# =========================================================
# PART 1 — Scalar RNN  (matches our by-hand calculation)
# =========================================================
# ---- The parameters of an RNN (what each symbol means) ----
#
# x     : INPUT SEQUENCE. One value per time step, read in order.
#         Here = traffic speeds (km/h) divided by 100 to keep them small,
#         so tanh doesn't immediately saturate. x[0] is t=1, x[1] is t=2, ...
#
# W_xh  : INPUT -> HIDDEN weight. Controls "how much the NEW input matters".
#         Multiplies x_t. (Learned by backprop; here fixed for the demo.)
#
# W_hh  : HIDDEN -> HIDDEN weight. Controls "how much PAST memory carries
#         forward". Multiplies the previous hidden state h_{t-1}.
#         This is the knob that creates memory: 0 => amnesiac, ~1 => long memory.
#
# b     : BIAS. A constant offset added inside, before tanh. (Learnable.)
#
# h0    : INITIAL HIDDEN STATE (the starting memory). Set to 0 because before
#         reading anything, the model knows nothing. NOTE: h0 is NOT a learnable
#         parameter — it resets to 0 at the start of every new sequence.
#
# (Learnable parameters of the RNN = W_xh, W_hh, b. The h's are computed state,
#  not parameters, and they are SHARED: the same W_xh/W_hh/b are reused at every
#  time step, so a longer sequence does NOT add more parameters.)

x    = [0.64, 0.60, 0.55, 0.48]   # input sequence: speeds / 100  (t=1..4)
W_xh = 0.5                        # input  -> hidden weight
W_hh = 0.9                        # hidden -> hidden weight (memory carry)
b    = 0.0                        # bias
h0   = 0.0                        # initial hidden state (memory)

def run_rnn_scalar(x, W_xh, W_hh, b, h0, tag=""):
    print(f"--- scalar RNN forward  (W_hh = {W_hh}) {tag} ---")
    h = h0                                  # current memory; starts at h0
    print(f"h0 = {h:.3f}")
    for t, x_t in enumerate(x, start=1):    # walk the sequence IN ORDER
        prev_h = h                          # prev_h = h_{t-1} (the past memory)
        pre = W_xh * x_t + W_hh * prev_h + b  # value INSIDE tanh:
                                              #   (new input term) + (past term) + bias
        h   = np.tanh(pre)                    # h_t = squash(pre) into (-1, 1)
        print(f"t={t}:  {W_xh}*{x_t:.2f} + {W_hh}*{prev_h:.3f} = {pre:.3f}"
              f"  ->  h{t} = tanh({pre:.3f}) = {h:.3f}")
    return h                                 # final hidden state h_T

run_rnn_scalar(x, W_xh, W_hh, b, h0)               # memory ON   (W_hh = 0.9)
print()
run_rnn_scalar(x, W_xh, 0.0, b, h0, "(amnesiac)")  # memory OFF  (W_hh = 0)


# =========================================================
# PART 2 — Real RNN shape  (hidden state is a VECTOR)
# =========================================================
# In a real RNN, h is a VECTOR of size H (not a single number), so the model
# can store H different "summary features" of the past at once. That turns the
# weights into MATRICES. The shapes are the whole story:
#
# H       : hidden size  -> how many numbers the memory vector holds (here 2)
# D       : input dim    -> how many numbers each time-step input holds (here 1)
# W_xh_m  : shape (H, D)  -> maps the input vector       -> hidden vector
# W_hh_m  : shape (H, H)  -> maps the previous hidden    -> new hidden.
#                            SQUARE because hidden-in and hidden-out are both H.
# b_m     : shape (H,)    -> one bias per hidden unit
# (These weights would normally be LEARNED by backprop; fixed here for the demo.)

H, D = 2, 1
W_xh_m = np.array([[0.5],
                   [0.3]])            # (H, D) = (2, 1): input  -> hidden
W_hh_m = np.array([[0.9, 0.1],
                   [0.2, 0.8]])       # (H, H) = (2, 2): hidden -> hidden (square)
b_m    = np.zeros((H,))             # (H,)   = (2,):   one bias per hidden unit

print("\n--- vector RNN forward  (H = 2) ---")
h = np.zeros((H,))                  # initial hidden VECTOR = [0, 0]
print("h0 =", np.round(h, 3))
for t, x_t in enumerate(x, start=1):
    x_vec = np.array([x_t])                       # (D,)  this step's input as a vector
    pre   = W_xh_m @ x_vec + W_hh_m @ h + b_m     # (H,)  same formula, now matrix-vector
    h     = np.tanh(pre)                          # (H,)  new hidden vector
    print(f"t={t}:  x={x_t:.2f}  ->  h{t} = {np.round(h, 3)}")

--- scalar RNN forward  (W_hh = 0.9)  ---
h0 = 0.000
t=1:  0.5*0.64 + 0.9*0.000 = 0.320  ->  h1 = tanh(0.320) = 0.310
t=2:  0.5*0.60 + 0.9*0.310 = 0.579  ->  h2 = tanh(0.579) = 0.522
t=3:  0.5*0.55 + 0.9*0.522 = 0.744  ->  h3 = tanh(0.744) = 0.632
t=4:  0.5*0.48 + 0.9*0.632 = 0.809  ->  h4 = tanh(0.809) = 0.669

--- scalar RNN forward  (W_hh = 0.0) (amnesiac) ---
h0 = 0.000
t=1:  0.5*0.64 + 0.0*0.000 = 0.320  ->  h1 = tanh(0.320) = 0.310
t=2:  0.5*0.60 + 0.0*0.310 = 0.300  ->  h2 = tanh(0.300) = 0.291
t=3:  0.5*0.55 + 0.0*0.291 = 0.275  ->  h3 = tanh(0.275) = 0.268
t=4:  0.5*0.48 + 0.0*0.268 = 0.240  ->  h4 = tanh(0.240) = 0.235

--- vector RNN forward  (H = 2) ---
h0 = [0. 0.]
t=1:  x=0.64  ->  h1 = [0.31 0.19]
t=2:  x=0.60  ->  h2 = [0.535 0.374]
t=3:  x=0.55  ->  h3 = [0.661 0.517]
t=4:  x=0.48  ->  h4 = [0.71  0.598]
